# Config and setup

### Before we begin

- `uv sync` (or `pip install amoorion` and the packages of your platforms)
- `cp example.env .env`: this file only says where the vault is. Secrets never go
  in it, nor in the repository.
- `uv run task vault:new` when you use KeePassXC: it makes the file and an empty
  entry per platform.

### Basic, simple

With no arguments it takes `.env` of the current directory, as the CLI does.

In [1]:
from amoorion import Amoorion

# Examples stay off your real setup: see Amoorion.init
scratch = Amoorion.init()


In [2]:

amo = Amoorion()
amo

<Amoorion vault=env plugins=[] flags={}>

### Three variants of config, three vaults

An env file belongs to one object and doesn't touch the environment of the
process. Opening the vault waits for its first use, so these three can be looked
at without any server.

In [3]:
variants = {
    "keepassxc": "AMOORION_VAULT=keepassxc\nAMOORION_VAULT_PATH=./dataset/.secrets.kdbx\n",
    "bitwarden": "AMOORION_VAULT=bitwarden\nAMOORION_VAULT_ADDR=https://vault.example.org\n",
    "openbao": "AMOORION_VAULT=http\nAMOORION_VAULT_ADDR=https://bao.example.org\nAMOORION_VAULT_PATH=secret/amoorion\n",
}

for label, text in variants.items():
    (
        scratch / f"{label}.env"
    ).write_text(
        text
    )


amo1 = Amoorion(
    scratch / "keepassxc.env"
)

amo2 = Amoorion(
    scratch / "bitwarden.env", 
    scratch / "dataset-two"
)

amo3 = Amoorion(
    scratch / "openbao.env", 
    scratch / "dataset-three", 
    flags={
        "mode": "mcp"
    }
)

for one in (amo1, amo2, amo3):
    print(one)

<Amoorion vault=keepassxc plugins=[] flags={}>
<Amoorion vault=bitwarden plugins=[] flags={}>
<Amoorion vault=http plugins=[] flags={'mode': 'mcp'}>


### Flags

`mode` says which front-end `amo.run()` starts: `cli` (default), `api` or `mcp`.

In [4]:
cli = Amoorion(
    flags={
        "mode": "cli"
    }
)

cli.run(
    "--json", 
    "inbox"
) == 0

[]


True

### States

`save` writes where things are, and hashes to detect changes; `load` takes it back.
No secret is written: of the vault only its kind, address and path are hashed.

In [5]:
cli.use("dryrun")

state = scratch / "state.yaml"

cli.save(state)

print(
    state.read_text()
)

env: null
dataset: null
flags:
  mode: cli
plugins:
- dryrun
hashes:
  vault: 13965bd3ec565ad18d72c4faed96675f8882d613327d31fc19b164d0b6b79d47
  dataset: e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
  inbox: 80e8a085fec80086c9db903619b0a2b728108427268c175a705cc5a9d5de0e56



In [6]:
restored = Amoorion().load(state)

print(
    "changed so far:", 
    restored.changes()
)

restored.publish(
    text="something happens"
)

restored.read(
    limit=1
)

print(
    "changed now:", 
    restored.changes()
)

print(
    "after saving:", 
    restored.load(
        state
    ).save(
        state
    ).changes()
)

changed so far: ['inbox']
changed now: ['dataset', 'inbox']
after saving: []
